# LIGO Glitch Classifier

Convolutional-network classifier for transient noise artifacts ("glitches")
in LIGO gravitational-wave detector data, benchmarked against the
[Gravity Spy](https://www.zooniverse.org/projects/zooniverse/gravity-spy)
citizen-science labeling project.

**Background.** LIGO detects gravitational waves by measuring sub-atomic
length changes in its interferometer arms, which makes the instrument
extremely susceptible to non-astrophysical noise transients from seismic
activity, scattered light, electronics, and mechanical resonances. These
glitches can mimic or mask real astrophysical signals, so classifying them
is a standard part of LIGO detector characterization. Gravity Spy is the
reference dataset and labeling scheme for this task: it represents each
glitch as a set of time-frequency spectrogram images (Q-transforms) at four
duration windows (0.5 s, 1 s, 2 s, 4 s), labeled by a combination of
citizen-science volunteers and a production ML classifier.

**Approach.** A ResNet18 backbone, fine-tuned from ImageNet weights, is
trained on the 1.0 s duration-view spectrograms from the Gravity Spy
training set. Two runs are reported: a baseline on 8 well-separated classes,
and a full run on the complete 22-class taxonomy. Predictions are then
benchmarked against Gravity Spy's own published ML labels and its volunteer
consensus labels for the same glitches.

**Repository:** https://github.com/amishi71/ligo-glitch-classifier

**Runtime:** requires a GPU (Runtime → Change runtime type → T4 GPU).

---
**Changelog (this version):**
- `benchmark.py`'s volunteer-label reader now uses PyTables to deserialize the
  pickled object block, instead of a hand-rolled h5py + manual `pickle.loads`
  reconstruction that was fragile and would hang/misbehave.
- Benchmark results now save **incrementally** — the ML-label merge is written
  to disk before the (slower) volunteer-label step runs, so an interrupt there
  never costs you the ML-label results too.
- Section 1 now clones/updates the repo robustly regardless of the notebook's
  current working directory, and writes the fixed `benchmark.py` to disk
  directly so the fix is guaranteed present even if you haven't pushed/pulled it.
- Google Drive is now optional. Drive's `mount()` was unreliable in testing, so
  the primary way to get your results off the runtime is now a plain zip
  download (Section 12) that doesn't depend on Drive at all.
- Section 4 (8-class baseline) is clearly marked optional/skippable — it's a
  sanity check, not required for the 22-class benchmark.

## 1. Environment setup

Clone the repository (or update it if already cloned) and install dependencies.

In [ ]:
import os

REPO_URL = "https://github.com/amishi71/ligo-glitch-classifier.git"
BASE_DIR = "/content"
REPO_DIR = os.path.join(BASE_DIR, "ligo-glitch-classifier")

%cd $BASE_DIR
if not os.path.isdir(REPO_DIR):
    !git clone $REPO_URL
else:
    print(f"{REPO_DIR} already exists -- skipping clone (the next cell pulls latest changes)")
%cd $REPO_DIR
!pwd


In [ ]:
%cd /content/ligo-glitch-classifier
!git pull


In [ ]:
!pip install -q -r requirements.txt


### 1b. Apply the fixed `benchmark.py` directly

Writes the corrected benchmark script straight to disk, overwriting whatever
`git pull` brought in. This guarantees the fix is present in this session
regardless of whether it's been pushed to the repo yet — one less thing to
depend on today.

In [ ]:
%%writefile src/benchmark.py
"""Benchmark a trained checkpoint against Gravity Spy ML and volunteer labels.

Joins the model's test-set predictions to the official Gravity Spy labels on
`gravityspy_id`:
  - ML-classification CSVs, one per detector+run: https://zenodo.org/records/5649212
  - Volunteer/ML consensus HDF5 (single file, `final_label` column):
    https://zenodo.org/records/5911227

Note: there is a *different* Zenodo record (13904422) that also has "volunteer
classifications" in its title, but it's a raw per-vote export keyed on a
Zooniverse `Subject_id`, not `gravityspy_id` -- it is not usable here without
a separate aggregation step, so this script expects the 5911227 file instead.
`download_data.py --volunteer-labels` already downloads the correct one.

Usage (after train.py has produced a checkpoint, and download_data.py has
pulled the label files):
    python src/benchmark.py \
        --checkpoint checkpoints/best_model.pt \
        --ml-labels data/labels/H1_O3a.csv data/labels/H1_O3b.csv \
        --volunteer-labels data/labels/retired_fulldata_min2_max50_ret0p9.hdf5

Either --ml-labels or --volunteer-labels can be omitted if you only have one.
Reading the volunteer HDF5 requires the `tables` package (pip install tables).

Results are written to <output-dir>/benchmark_results.csv incrementally: the
ML-label merge is saved to disk as soon as it completes, before the (slower,
heavier) volunteer-label step starts, so an interrupt or crash during the
volunteer step never costs you the ML-label results too.
"""
import argparse
import os
import time

import numpy as np
import pandas as pd
import tables
import torch

from dataset import get_dataloaders
from model import build_model


def load_ml_labels(paths):
    frames = [pd.read_csv(p) for p in paths]
    df = pd.concat(frames, ignore_index=True)
    df = df[["gravityspy_id", "ml_label"]].drop_duplicates("gravityspy_id")
    return df


def _decode_if_bytes(arr):
    """String columns round-tripped through a pickled numpy object block can
    come back as either `str` or `bytes` elements depending on how they were
    originally written; normalize to `str`."""
    arr = np.asarray(arr, dtype=object)
    if arr.size and isinstance(arr.flat[0], bytes):
        return np.array([x.decode() for x in arr])
    return arr


def _available_mem_gb():
    """Best-effort free-RAM read from /proc/meminfo (Linux/Colab only).
    Returns None rather than raising if it's not available -- this is a
    diagnostic nicety, not something the pipeline should ever depend on."""
    try:
        with open("/proc/meminfo") as f:
            for line in f:
                if line.startswith("MemAvailable"):
                    return int(line.split()[1]) / 1e6  # kB -> GB
    except Exception:
        return None


def _read_fixed_format_columns(path, group, columns):
    """Reads a set of named columns out of a pandas 'fixed'-format HDF5
    store, without loading the (much larger) numeric blocks.

    Background: pandas' 'fixed' HDF5 format groups a DataFrame's columns
    into blocks by dtype. Numeric blocks land as plain dense arrays and are
    cheap to read with either h5py or PyTables. Object/string blocks are
    different: pandas pickles the *entire* block (every string column,
    every row) as a single object and writes it as one row of a PyTables
    `VLArray` with `ObjectAtom`.

    Reading that block via PyTables (as this function does) hands back the
    already-deserialized array directly -- PyTables wrote it, so PyTables
    reads it back correctly. Reading it via raw h5py instead only gives you
    the opaque serialized bytes, and reconstructing those by hand (manual
    `pickle.loads`, guessing at `.tobytes()` and array orientation) means
    guessing at an internal, version-dependent pandas/PyTables encoding --
    which is what an earlier version of this function did, and why it used
    to hang/misbehave.

    Reads every requested column in a single pass over the file (no
    re-opening or re-unpickling the same block once per column).

    Returns a dict {column_name: 1-D array}.
    """
    remaining = set(columns)
    found = {}
    with tables.open_file(path, mode="r") as h5file:
        group_node = h5file.get_node("/" + group)
        i = 0
        while remaining and hasattr(group_node, f"block{i}_items"):
            items = [c.decode() if isinstance(c, bytes) else c
                     for c in getattr(group_node, f"block{i}_items")[:]]
            wanted_here = remaining & set(items)
            if wanted_here:
                values_node = getattr(group_node, f"block{i}_values")
                block = values_node[:]
                if block.dtype == object and block.shape[0] == 1:
                    # whole block was pickled together as one VLArray row;
                    # PyTables has already unpickled it for us here
                    block = np.asarray(block[0])
                for col in wanted_here:
                    col_idx = items.index(col)
                    if block.ndim == 1:
                        col_vals = block
                    elif block.shape[1] == len(items):
                        col_vals = block[:, col_idx]
                    elif block.shape[0] == len(items):
                        col_vals = block[col_idx]
                    else:
                        raise ValueError(
                            f"Unexpected block shape {block.shape} for items {items}"
                        )
                    found[col] = _decode_if_bytes(col_vals)
                remaining -= wanted_here
            i += 1
    if remaining:
        raise KeyError(f"Column(s) {remaining} not found in {group!r} of {path}")
    return found


def load_volunteer_labels(path):
    """Loads the pre-aggregated volunteer+ML consensus dataset.

    Expects the HDF5 file from https://zenodo.org/records/5911227
    (`retired_fulldata_min2_max50_ret0p9.hdf5`), which has one row per glitch
    with `gravityspy_id` and `final_label` columns already computed -- no
    per-vote aggregation needed on our end.

    The file is ~1GB, mostly from ~20 per-class ML confidence columns we
    don't need. A plain `pd.read_hdf(path)` materializes the whole table --
    every column, every row -- in memory before we get to subset it, which
    is exactly the kind of spike that gets a process silently OOM-killed on
    a constrained runtime like Colab's free tier (no Python traceback, the
    process just disappears -- indistinguishable from a hang from the
    outside). So this reads only the two needed columns directly.

    If the file is in pytables 'table' format, `HDFStore.select(...,
    chunksize=...)` streams it row-chunk by row-chunk with `columns=`
    dropping everything else before each chunk is held in memory. If it's
    in 'fixed' format (no chunked-read support at all), falls back to
    `_read_fixed_format_columns`, which reads only the two needed columns
    via PyTables directly rather than the whole table.
    """
    size_gb = os.path.getsize(path) / 1e9
    mem_before = _available_mem_gb()
    mem_str = f", {mem_before:.2f} GB RAM available" if mem_before is not None else ""
    print(f"Loading volunteer consensus file ({size_gb:.2f} GB){mem_str}...")
    t0 = time.time()

    with pd.HDFStore(path, mode="r") as store:
        storer = store.get_storer("image_db")
        is_table = storer.is_table
        group_name = storer.group._v_pathname.lstrip("/")

    if is_table:
        chunksize = 20000
        print(f"  Table format detected -- streaming in chunks of {chunksize}...")
        chunks = []
        with pd.HDFStore(path, mode="r") as store:
            for i, chunk in enumerate(
                store.select("image_db", columns=["gravityspy_id", "final_label"], chunksize=chunksize)
            ):
                chunks.append(chunk)
                print(f"  ...read chunk {i + 1} ({sum(len(c) for c in chunks)} rows so far, "
                      f"{time.time() - t0:.0f}s elapsed)")
        df = pd.concat(chunks, ignore_index=True)
    else:
        print("  Fixed format detected -- reading only the gravityspy_id and final_label "
              "columns directly (via PyTables), skipping the wide numeric blocks that make "
              "up most of the file.")
        cols = _read_fixed_format_columns(path, group_name, ["gravityspy_id", "final_label"])
        elapsed = time.time() - t0
        mem_after = _available_mem_gb()
        mem_str = f", {mem_after:.2f} GB RAM available" if mem_after is not None else ""
        print(f"  ...read {len(cols['gravityspy_id'])} rows in {elapsed:.1f}s{mem_str}, "
              "building DataFrame...")
        df = pd.DataFrame({"gravityspy_id": cols["gravityspy_id"], "final_label": cols["final_label"]})

    print(f"Loaded {len(df)} rows in {time.time() - t0:.1f}s")
    df = df.rename(columns={"final_label": "volunteer_label"})
    return df[["gravityspy_id", "volunteer_label"]].drop_duplicates("gravityspy_id")


def main():
    parser = argparse.ArgumentParser(description="Benchmark predictions against Gravity Spy labels.")
    parser.add_argument("--data-path", default="data/raw/trainingsetv1d1.h5")
    parser.add_argument("--checkpoint", default="checkpoints/best_model.pt")
    parser.add_argument("--duration", default="1.0", choices=["0.5", "1.0", "2.0", "4.0"])
    parser.add_argument("--split-train", default="train")
    parser.add_argument("--split-val", default="validation")
    parser.add_argument("--split-test", default="test")
    parser.add_argument("--batch-size", type=int, default=64)
    parser.add_argument("--image-size", type=int, default=224)
    parser.add_argument("--ml-labels", nargs="*", default=[], help="path(s) to Gravity Spy ML label CSVs")
    parser.add_argument("--volunteer-labels", default=None, help="path to the volunteer consensus HDF5")
    parser.add_argument("--output-dir", default="outputs")
    args = parser.parse_args()

    if not args.ml_labels and not args.volunteer_labels:
        raise SystemExit("Pass at least one of --ml-labels or --volunteer-labels.")

    device = "cuda" if torch.cuda.is_available() else "cpu"
    ckpt = torch.load(args.checkpoint, map_location=device)
    classes = ckpt["classes"]

    split_names = {"train": args.split_train, "val": args.split_val, "test": args.split_test}
    _, _, test_loader, _, _, test_ds = get_dataloaders(
        args.data_path, split_names, classes, args.duration, args.batch_size, image_size=args.image_size,
    )

    model = build_model(len(classes)).to(device)
    model.load_state_dict(ckpt["model_state"])
    model.eval()

    all_preds = []
    with torch.no_grad():
        for x, _ in test_loader:
            x = x.to(device)
            all_preds.extend(model(x).argmax(1).cpu().numpy())

    # test_ds.index lists (label, gravityspy_id) in the exact order the
    # DataLoader iterated it (shuffle=False for val/test loaders), so we can
    # zip predictions back onto ids positionally.
    true_labels = [label for label, _ in test_ds.index]
    gids = [gid for _, gid in test_ds.index]
    pred_labels = [classes[p] for p in all_preds]

    results = pd.DataFrame({
        "gravityspy_id": gids,
        "true_label": true_labels,
        "model_pred": pred_labels,
    })

    os.makedirs(args.output_dir, exist_ok=True)
    out_path = os.path.join(args.output_dir, "benchmark_results.csv")

    if args.ml_labels:
        ml_df = load_ml_labels(args.ml_labels)
        results = results.merge(ml_df, on="gravityspy_id", how="left")
        matched = results["ml_label"].notna()
        print(f"ML labels matched: {matched.sum()}/{len(results)} test glitches found in the ML CSV(s) you passed")
        if matched.any():
            agree = (results.loc[matched, "model_pred"] == results.loc[matched, "ml_label"]).mean()
            ref_acc = (results.loc[matched, "ml_label"] == results.loc[matched, "true_label"]).mean()
            print(f"  Your model vs Gravity Spy ML label agreement: {agree:.4f}")
            print(f"  Gravity Spy ML label vs training-set ground truth: {ref_acc:.4f}  (sanity check / reference point)")
        else:
            print("  No matches -- likely means the CSV(s) you passed don't cover the detector/run your "
                  "test-set glitches came from. Check which H1_*/L1_* files you downloaded.")
        results.to_csv(out_path, index=False)
        print(f"  Saved intermediate results (ML labels) to {out_path}")

    if args.volunteer_labels:
        try:
            vol_df = load_volunteer_labels(args.volunteer_labels)
        except Exception:
            print(f"\nVolunteer-label loading failed or was interrupted. "
                  f"ML-label results (if any) are still safely saved at {out_path}.")
            raise
        results = results.merge(vol_df, on="gravityspy_id", how="left")
        matched = results["volunteer_label"].notna()
        print(f"Volunteer labels matched: {matched.sum()}/{len(results)} test glitches found in the volunteer file")
        if matched.any():
            agree = (results.loc[matched, "model_pred"] == results.loc[matched, "volunteer_label"]).mean()
            print(f"  Your model vs volunteer-consensus label agreement: {agree:.4f}")
        results.to_csv(out_path, index=False)

    print("\nSaved full per-glitch comparison table to", out_path)

    mismatches = results[results["model_pred"] != results["true_label"]]
    print(f"{len(mismatches)} model predictions disagree with the training-set label -- "
          f"inspect these rows first when doing error analysis.")


if __name__ == "__main__":
    main()


## 2. Data acquisition

Downloads the Gravity Spy training set (`trainingsetv1d1.h5`, ~2.8 GB) from
Zenodo. This file contains pre-rendered spectrogram images for every
labeled glitch, organized by class and by train/validation/test split.

Label CSVs for benchmarking (Section 9) are downloaded separately later,
since they are only needed after a model has been trained.

In [ ]:
!python src/download_data.py --training-set


## 3. Dataset structure check

The HDF5 file's internal split names (train / validation / test) are
verified explicitly before training, since `dataset.py` requires an exact
string match against whatever split names are actually stored in the file.
A mismatch here fails silently — it produces an empty dataset rather than
an error — so this check is run as a first step rather than discovered
during training.

In [ ]:
import h5py

with h5py.File("data/raw/trainingsetv1d1.h5", "r") as f:
    class_names = list(f.keys())
    # any class works as a probe since split names are consistent across classes
    probe_class = "Blip" if "Blip" in class_names else class_names[0]
    split_names = list(f[probe_class].keys())

print(f"Classes ({len(class_names)}): {class_names}")
print(f"Splits: {split_names}")

# dataset.py defaults to ['train', 'validation', 'test']; if the file uses
# different names, override with --split-train/--split-val/--split-test
# in the train.py and evaluate.py calls below.
assert split_names, "No splits found under the probe class — check the HDF5 file."


## 4. Baseline: 8-class training *(optional)*

Trains on a subset of 8 well-separated glitch classes (`Blip`, `Chirp`,
`Koi_Fish`, `Low_Frequency_Burst`, `Power_Line`, `Scattered_Light`,
`Violin_Mode`, `Whistle`) as a fast sanity check before committing to the
full 22-class run. ResNet18, ImageNet-pretrained, 15 epochs, class-weighted
cross-entropy loss to account for per-class sample count differences.
Checkpoints are saved to `checkpoints/` whenever validation accuracy improves.

**This section is optional.** It is not required for the 22-class benchmark
in Section 10 — if you're short on time, skip straight to Section 5.

In [ ]:
!python src/train.py --epochs 15


Test-set accuracy, macro-averaged F1, and a per-class precision/recall/F1 report.

In [ ]:
!python src/evaluate.py --checkpoint checkpoints/best_model.pt


Confusion matrix for the 8-class baseline:

In [ ]:
from IPython.display import Image, display

display(Image("outputs/confusion_matrix.png"))


## 5. Full training: all 22 classes

The 8-class baseline confirms the pipeline works, but is not directly
comparable to Gravity Spy's published results, which cover the complete
taxonomy. This run trains on all 22 classes and saves to a separate
checkpoint directory so the baseline model (if you ran Section 4) is preserved.

In [ ]:
!python src/train.py --all-classes --epochs 15 --checkpoint-dir checkpoints_22class


## 6. Full-model evaluation

In [ ]:
!python src/evaluate.py --checkpoint checkpoints_22class/best_model.pt --output-dir outputs_22class


In [ ]:
from IPython.display import Image, display

display(Image("outputs_22class/confusion_matrix.png"))


## 7. Gravity Spy label download

Downloads the official Gravity Spy ML-classification files and the
volunteer+ML consensus labels for benchmarking (Section 10).

The ML label files are split by detector and observing run, and the
training-set HDF5 doesn't record which run each glitch came from, so
`--ml-labels all` downloads all eight (H1_O1, H1_O2, H1_O3a, H1_O3b, L1_O1,
L1_O2, L1_O3a, L1_O3b) to guarantee coverage rather than guessing. Section
10 prints the match rate against whatever files were downloaded, as a
diagnostic.

The volunteer-labels file is a single pre-aggregated HDF5
(`retired_fulldata_min2_max50_ret0p9.hdf5`) with one row per glitch,
`gravityspy_id`, and a `final_label` (the combined volunteer + ML consensus)
already computed.

In [ ]:
!python src/download_data.py --ml-labels all --volunteer-labels


## 8. Volunteer-labels file structure (diagnostic)

Confirms this file is in pandas' 'fixed' HDF5 format (not 'table'), which
is why `benchmark.py` reads it via the PyTables-based column reader rather
than a plain `pd.read_hdf()` or chunked `HDFStore.select()`.

In [ ]:
import h5py

with h5py.File('data/labels/retired_fulldata_min2_max50_ret0p9.hdf5', 'r') as f:
    def report(name, obj):
        if isinstance(obj, h5py.Dataset):
            try:
                storage_bytes = obj.id.get_storage_size()
            except Exception:
                storage_bytes = None
            print(f"{name:30s} shape={str(obj.shape):15s} dtype={str(obj.dtype):10s} storage_bytes={storage_bytes}")
    f.visititems(report)


## 9. Quick isolated test of the volunteer-label reader

Before running the full benchmark (which repeats model inference and the
ML-label merge every time), test just the part that was previously
hanging/failing, on its own. This is cheap to re-run and prints free-RAM
before/after, so if it ever dies silently again, you'll be able to tell
immediately whether it's memory pressure. **Let this run to completion
without interrupting it** — it can genuinely take a couple of minutes to
unpickle ~809k rows, that's not a hang.

In [ ]:
import sys, time
sys.path.insert(0, "src")
from benchmark import load_volunteer_labels

t0 = time.time()
df = load_volunteer_labels("data/labels/retired_fulldata_min2_max50_ret0p9.hdf5")
print(f"done in {time.time() - t0:.1f}s, {len(df)} rows")
df.head()


## 10. Benchmark against Gravity Spy labels

Joins the trained model's test-set predictions to the official Gravity Spy
labels on `gravityspy_id` (the shared identifier across the training-set
HDF5, the ML label CSVs, and the volunteer+ML consensus HDF5) and reports:

- Agreement between the model's predictions and Gravity Spy's own ML
  classifier's labels
- Agreement between the model's predictions and the volunteer+ML consensus
  labels (`final_label`)
- A reference point: agreement between Gravity Spy's ML label and the
  training-set ground-truth label

A full per-glitch comparison table is written to
`outputs_22class/benchmark_results.csv` for error analysis — in particular,
cases where the model disagrees with the volunteer label are often
genuinely ambiguous glitches rather than model errors, and are worth
inspecting individually.

Results are saved after the ML-label step *and* after the volunteer-label
step, so even if this cell is interrupted partway, whatever completed is
already on disk in `outputs_22class/benchmark_results.csv`.

In [ ]:
!python src/benchmark.py \
    --checkpoint checkpoints_22class/best_model.pt \
    --ml-labels data/labels/H1_O1.csv data/labels/H1_O2.csv data/labels/H1_O3a.csv data/labels/H1_O3b.csv \
                data/labels/L1_O1.csv data/labels/L1_O2.csv data/labels/L1_O3a.csv data/labels/L1_O3b.csv \
    --volunteer-labels data/labels/retired_fulldata_min2_max50_ret0p9.hdf5 \
    --output-dir outputs_22class


## 11. Review results

A quick look at the saved comparison table — full per-glitch detail is in
`outputs_22class/benchmark_results.csv`.

In [ ]:
import pandas as pd

results = pd.read_csv("outputs_22class/benchmark_results.csv")
print(f"{len(results)} rows")
display(results.head())

mismatches = results[results["model_pred"] != results["true_label"]]
print(f"\n{len(mismatches)} model predictions disagree with the training-set label:")
display(mismatches.head(10))


## 12. Save your results (no Google Drive required)

Zips the checkpoint, evaluation outputs, and benchmark results, and
downloads the zip straight to your machine through the browser. This is the
primary way to get your work off this runtime today — Drive's `mount()`
was unreliable earlier in this project, so this path avoids it entirely.

In [ ]:
import shutil, zipfile, os

shutil.make_archive("ligo_glitch_results", "zip", ".", "checkpoints_22class")
# Add outputs_22class into the same archive
with zipfile.ZipFile("ligo_glitch_results.zip", "a") as zf:
    for root, _, files_ in os.walk("outputs_22class"):
        for fname in files_:
            fpath = os.path.join(root, fname)
            zf.write(fpath, fpath)

print("Archive ready:", os.path.getsize("ligo_glitch_results.zip") / 1e6, "MB")

from google.colab import files
files.download("ligo_glitch_results.zip")


## 13. Google Drive backup *(optional — skip if it gives you trouble)*

Colab sessions are ephemeral, and Drive is the normal way to persist files
across sessions. In this project's testing, `drive.mount()` failed
repeatedly with a generic `mount failed` error even after `force_remount`
and a runtime restart — if that happens to you too, don't spend time on it;
Section 12 above already got your results off the runtime without it.

If you do want to try: `Runtime → Disconnect and delete runtime` for a
genuinely fresh VM, then run this cell as the *first* thing in the new
session, before cloning anything else, tends to be the most reliable order.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
import shutil

backup_dir = "/content/drive/MyDrive/ligo_glitch_backup"
os.makedirs(backup_dir, exist_ok=True)

for folder in ["checkpoints_22class", "outputs_22class", "src/benchmark.py"]:
    dest = os.path.join(backup_dir, os.path.basename(folder))
    if os.path.isdir(folder):
        if os.path.exists(dest):
            shutil.rmtree(dest)
        shutil.copytree(folder, dest)
    else:
        shutil.copy(folder, dest)
    print(f"Backed up {folder} -> {dest}")
